# Notebook 01 — Data Exploration

**Project:** Cognitive Fire Defense Pipeline — AIN7601  
**Purpose:** Validate both datasets, explore class distributions, visualize sample images

## Sections
1. Environment setup
2. Dataset A — Forest Fire (Drone/UAV)
3. Dataset B — Fairdata (Static/Watchtower)
4. Class distribution analysis
5. Sample image visualization

In [ ]:
# ── 0. Install dependencies (Colab only) ──────────────────────────────────
# Uncomment if running on Google Colab
# !pip install kagglehub ultralytics pycocotools albumentations --quiet

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Project root
ROOT = Path("..")
DATA_A = ROOT / "data" / "dataset-a"
DATA_B = ROOT / "data" / "dataset-b"

print(f"Dataset A path: {DATA_A.resolve()}")
print(f"Dataset B path: {DATA_B.resolve()}")

In [ ]:
# ── 1. Download Dataset A (run once) ─────────────────────────────────────
import kagglehub
import shutil

if not any((DATA_A / "raw").iterdir() if (DATA_A / "raw").exists() else []):
    path = kagglehub.dataset_download("alik05/forest-fire-dataset")
    shutil.copytree(path, DATA_A / "raw", dirs_exist_ok=True)
    print(f"Downloaded to {DATA_A / 'raw'}")
else:
    print("Dataset A already present.")

In [ ]:
# ── 2. Count Dataset A images & labels ───────────────────────────────────
raw_a = DATA_A / "raw"

for split in ["train", "valid", "test"]:
    img_dir = raw_a / split / "images"
    lbl_dir = raw_a / split / "labels"
    if img_dir.exists():
        imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
        lbls = list(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []
        empty = sum(1 for l in lbls if l.stat().st_size == 0)
        print(f"  {split}: {len(imgs)} images | {len(lbls)} labels | {empty} empty labels")

In [ ]:
# ── 3. Visualize Dataset A samples ───────────────────────────────────────
CLASSES = ["fire", "smoke"]
COLORS  = ["#FF4444", "#AAAAAA"]

def show_yolo_sample(img_path, lbl_path, ax, title=""):
    img = np.array(Image.open(img_path))
    h, w = img.shape[:2]
    ax.imshow(img)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
    if lbl_path.exists() and lbl_path.stat().st_size > 0:
        for line in lbl_path.read_text().strip().splitlines():
            cls, cx, cy, bw, bh = map(float, line.split())
            x = (cx - bw/2) * w
            y = (cy - bh/2) * h
            rect = patches.Rectangle((x, y), bw*w, bh*h,
                                      linewidth=2, edgecolor=COLORS[int(cls)],
                                      facecolor="none")
            ax.add_patch(rect)
            ax.text(x, y-4, CLASSES[int(cls)], color=COLORS[int(cls)], fontsize=7)

img_dir = raw_a / "train" / "images"
lbl_dir = raw_a / "train" / "labels"
samples = list(img_dir.glob("*.jpg"))[:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_p in zip(axes.flat, samples):
    lbl_p = lbl_dir / (img_p.stem + ".txt")
    show_yolo_sample(img_p, lbl_p, ax, title=img_p.name[:20])
plt.suptitle("Dataset A — Sample Annotations (YOLO TXT)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(ROOT / "paper" / "figures" / "dataset_a_samples.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to paper/figures/dataset_a_samples.png")

In [ ]:
# ── 4. Class distribution plot ────────────────────────────────────────────
class_counts = {"fire": 0, "smoke": 0}

for lbl_path in lbl_dir.glob("*.txt"):
    for line in lbl_path.read_text().strip().splitlines():
        if line:
            cls = int(line.split()[0])
            class_counts[CLASSES[cls]] += 1

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(class_counts.keys(), class_counts.values(), color=["#FF4444", "#888888"])
ax.bar_label(bars)
ax.set_title("Dataset A — Class Distribution (Train)")
ax.set_ylabel("Number of instances")
plt.tight_layout()
plt.savefig(ROOT / "paper" / "figures" / "dataset_a_class_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print(class_counts)